# 06 - Leakage-Safe ResNet-18 Fine-Tuning in Google Colab

This notebook fine-tunes an ImageNet-pretrained ResNet-18 on the roll-only fingerprint dataset. It uses the existing subject-level train/validation/test split and stops immediately if any subject or image leaks between splits.

Training has two stages: classifier-head warm-up, then full-network fine-tuning. The test set is evaluated only once after model selection is complete.

## 1. Runtime setup

In Colab, select **Runtime → Change runtime type → T4 GPU** before running all cells.

In [ ]:
!pip install -q pandas numpy matplotlib scikit-learn torch torchvision tqdm

In [ ]:
from pathlib import Path
import json, random, time, zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, f1_score,
)
from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    raise RuntimeError("GPU not detected. Select a T4 GPU in Colab before training.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Mount Drive, validate the ZIP, and load all 2,312 records

In [ ]:
drive.mount("/content/drive")

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/dermatoglyphic_project")
PACKAGE_PATH = DRIVE_PROJECT_DIR / "roll_cnn_colab_package.zip"
WORK_DIR = Path("/content/dermatoglyphic_resnet18")
DATA_DIR = WORK_DIR / "data"
MODEL_DIR = DRIVE_PROJECT_DIR / "models"
FIGURE_DIR = DRIVE_PROJECT_DIR / "figures"
for directory in (DATA_DIR, MODEL_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

required_files = {
    "roll_cnn_160x160_dataset.npz", "roll_cnn_metadata.csv",
    "label_mapping.json", "README_colab_package.txt",
}
if not PACKAGE_PATH.exists():
    raise FileNotFoundError(PACKAGE_PATH)
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("The dataset ZIP is corrupt.")
    missing = required_files.difference(archive.namelist())
    if missing:
        raise FileNotFoundError(f"Missing package files: {sorted(missing)}")
    archive.extractall(DATA_DIR)
print("Package integrity check passed:", PACKAGE_PATH)

In [ ]:
with np.load(DATA_DIR / "roll_cnn_160x160_dataset.npz", allow_pickle=True) as prepared:
    images = prepared["images"]
    labels = prepared["labels"]
    splits = prepared["splits"].astype(str)
    label_names = prepared["label_names"].tolist()
metadata = pd.read_csv(
    DATA_DIR / "roll_cnn_metadata.csv",
    dtype={"subject_id": "string", "finger_position": "string"},
)

package_counts = {"train": 1620, "validation": 344, "test": 348}
assert images.shape == (2312, 160, 160)
assert images.dtype == np.uint8
assert len(labels) == len(splits) == len(metadata) == 2312
assert not metadata.isna().any().any()
assert np.array_equal(labels, metadata["label_id"].to_numpy(np.int64))
assert np.array_equal(splits, metadata["split"].to_numpy(str))
assert pd.Series(splits).value_counts().to_dict() == package_counts
assert metadata["png_path"].is_unique
print("Dataset validation passed.")
print("Images:", images.shape, "Classes:", label_names)
print("Original package split rows:", package_counts)

## 3. Mandatory leakage audit

A subject may have several fingers, devices, or captures. Therefore splitting individual images would leak identity-specific ridge characteristics.

The previous notebook already exposed its old test results. Reusing that test set to judge this improved model would bias the comparison. This notebook creates a fresh deterministic holdout from subjects that were **not** in the old test set. Previously evaluated test subjects are assigned to training only.

In [ ]:
old_splits = splits.copy()
old_test_subjects = set(metadata.loc[metadata["split"].eq("test"), "subject_id"])
never_tested_subjects = sorted(set(metadata["subject_id"]) - old_test_subjects)
rng = np.random.default_rng(SEED)
rng.shuffle(never_tested_subjects)

# Lock a new subject allocation before training. All previously tested subjects are train-only.
fresh_test_subjects = set(never_tested_subjects[:30])
validation_subjects = set(never_tested_subjects[30:60])
train_subjects = set(never_tested_subjects[60:]) | old_test_subjects
split_v2 = np.select(
    [metadata["subject_id"].isin(train_subjects),
     metadata["subject_id"].isin(validation_subjects),
     metadata["subject_id"].isin(fresh_test_subjects)],
    ["train", "validation", "test"], default="UNASSIGNED",
)
assert "UNASSIGNED" not in split_v2
splits = split_v2
metadata["original_split"] = metadata["split"]
metadata["split"] = splits
expected_counts = pd.Series(splits).value_counts().to_dict()

subject_sets = {
    split: set(metadata.loc[metadata["split"].eq(split), "subject_id"])
    for split in ("train", "validation", "test")
}
assert subject_sets["train"].isdisjoint(subject_sets["validation"])
assert subject_sets["train"].isdisjoint(subject_sets["test"])
assert subject_sets["validation"].isdisjoint(subject_sets["test"])
assert old_test_subjects.issubset(subject_sets["train"])
assert old_test_subjects.isdisjoint(subject_sets["validation"])
assert old_test_subjects.isdisjoint(subject_sets["test"])
split_manifest = metadata[["subject_id", "split"]].drop_duplicates().sort_values(["split", "subject_id"])
split_manifest.to_csv(DRIVE_PROJECT_DIR / "resnet18_subject_split_manifest.csv", index=False)

print("LEAKAGE AUDIT PASSED — fresh test set locked before training")
print("Unique subjects:", {k: len(v) for k, v in subject_sets.items()})
print("Fresh split rows:", expected_counts)
display(pd.crosstab(metadata["broad_class"], metadata["split"])[["train", "validation", "test"]])

## 4. Datasets and augmentation

Only training images are augmented. Validation and test transforms are deterministic. Horizontal flipping is intentionally excluded because it can alter left/right loop orientation.

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 2
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.025, 0.025), scale=(0.97, 1.03)),
    transforms.ColorJitter(brightness=0.10, contrast=0.15),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class FingerprintDataset(Dataset):
    def __init__(self, images, labels, indices, transform):
        self.images, self.labels = images, labels
        self.indices, self.transform = np.asarray(indices), transform
    def __len__(self): return len(self.indices)
    def __getitem__(self, item):
        index = self.indices[item]
        return self.transform(self.images[index]), int(self.labels[index]), int(index)

indices = {split: np.flatnonzero(splits == split) for split in expected_counts}
datasets = {
    "train": FingerprintDataset(images, labels, indices["train"], train_transform),
    "validation": FingerprintDataset(images, labels, indices["validation"], eval_transform),
    "test": FingerprintDataset(images, labels, indices["test"], eval_transform),
}
generator = torch.Generator().manual_seed(SEED)
loaders = {
    "train": DataLoader(datasets["train"], BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                        pin_memory=True, persistent_workers=True, generator=generator),
    "validation": DataLoader(datasets["validation"], BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                             pin_memory=True, persistent_workers=True),
    "test": DataLoader(datasets["test"], BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                       pin_memory=True, persistent_workers=True),
}
print({k: len(v) for k, v in datasets.items()})

## 5. Pretrained ResNet-18 and milder class weighting

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Sequential(nn.Dropout(0.35), nn.Linear(model.fc.in_features, len(label_names)))
model = model.to(device)

train_counts = np.bincount(labels[indices["train"]], minlength=len(label_names))
raw_weights = len(indices["train"]) / (len(label_names) * train_counts)
class_weights = torch.tensor(np.sqrt(raw_weights), dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
print("Train counts:", dict(zip(label_names, train_counts.tolist())))
print("Square-root weights:", dict(zip(label_names, class_weights.cpu().tolist())))
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    total_loss, all_true, all_pred = 0.0, [], []
    for batch_images, batch_labels, _ in loader:
        batch_images = batch_images.to(device, non_blocking=True)
        batch_labels = batch_labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
                logits = model(batch_images)
                loss = criterion(logits, batch_labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * batch_images.size(0)
        all_true.extend(batch_labels.detach().cpu().numpy())
        all_pred.extend(logits.argmax(1).detach().cpu().numpy())
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(all_true, all_pred),
        "macro_f1": f1_score(all_true, all_pred, average="macro"),
    }

def fit_stage(stage, epochs, optimizer, scheduler, history, best, patience):
    scaler = torch.amp.GradScaler("cuda")
    stale_epochs = 0
    for local_epoch in range(1, epochs + 1):
        started = time.perf_counter()
        train_metrics = run_epoch(model, loaders["train"], optimizer, scaler)
        val_metrics = run_epoch(model, loaders["validation"])
        scheduler.step(val_metrics["macro_f1"])
        elapsed = time.perf_counter() - started
        row = {"stage": stage, "stage_epoch": local_epoch, "seconds": elapsed,
               **{f"train_{k}": v for k, v in train_metrics.items()},
               **{f"validation_{k}": v for k, v in val_metrics.items()},
               "learning_rate": optimizer.param_groups[0]["lr"]}
        history.append(row)
        improved = val_metrics["macro_f1"] > best["macro_f1"] + 0.002
        if improved:
            best.update(macro_f1=val_metrics["macro_f1"], row=len(history) - 1)
            torch.save({"model_state": model.state_dict(), "label_names": label_names,
                        "best_validation_macro_f1": best["macro_f1"], "seed": SEED}, best["path"])
            stale_epochs = 0
        else:
            stale_epochs += 1
        print(f"{stage} {local_epoch:02d}/{epochs} | {elapsed:5.1f}s | "
              f"train f1={train_metrics['macro_f1']:.3f} | val f1={val_metrics['macro_f1']:.3f} | "
              f"val acc={val_metrics['accuracy']:.3f} | lr={optimizer.param_groups[0]['lr']:.2e}")
        if stale_epochs >= patience:
            print(f"Early stopping after {patience} epochs without meaningful improvement.")
            break
    return history, best

## 6. Two-stage training

This is intentionally more substantial than the earlier tiny CNN: up to 5 head-only epochs followed by up to 30 full-network epochs. Runtime depends on the Colab GPU and early stopping; timing is printed for every epoch.

In [ ]:
BEST_MODEL_PATH = MODEL_DIR / "resnet18_roll_best.pt"
HISTORY_PATH = DRIVE_PROJECT_DIR / "resnet18_training_history.csv"
history = []
best = {"macro_f1": -1.0, "row": None, "path": BEST_MODEL_PATH}

# Stage 1: train only the new classifier head.
for parameter in model.parameters(): parameter.requires_grad = False
for parameter in model.fc.parameters(): parameter.requires_grad = True
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=2)
history, best = fit_stage("head", 5, optimizer, scheduler, history, best, patience=5)

# Stage 2: unfreeze and fine-tune the complete network with a small learning rate.
for parameter in model.parameters(): parameter.requires_grad = True
optimizer = torch.optim.AdamW([
    {"params": model.layer1.parameters(), "lr": 1e-5},
    {"params": model.layer2.parameters(), "lr": 2e-5},
    {"params": model.layer3.parameters(), "lr": 4e-5},
    {"params": model.layer4.parameters(), "lr": 8e-5},
    {"params": model.fc.parameters(), "lr": 2e-4},
], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.3, patience=3)
history, best = fit_stage("full", 30, optimizer, scheduler, history, best, patience=8)

history_table = pd.DataFrame(history)
history_table.to_csv(HISTORY_PATH, index=False)
print("Best validation macro F1:", round(best["macro_f1"], 4))
print("Saved:", BEST_MODEL_PATH)
display(history_table)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.arange(1, len(history_table) + 1)
axes[0].plot(x, history_table["train_loss"], label="train")
axes[0].plot(x, history_table["validation_loss"], label="validation")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(x, history_table["train_accuracy"], label="train")
axes[1].plot(x, history_table["validation_accuracy"], label="validation")
axes[1].set_title("Accuracy"); axes[1].legend()
axes[2].plot(x, history_table["train_macro_f1"], label="train")
axes[2].plot(x, history_table["validation_macro_f1"], label="validation")
axes[2].set_title("Macro F1"); axes[2].legend()
for axis in axes: axis.set_xlabel("Completed epoch")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "resnet18_training_history.png", dpi=160)
plt.show()

## 7. One-time final test evaluation

Do not use these test results to tune the model. Any later modelling decision must use the validation set and requires a fresh untouched test set for a strictly unbiased final estimate.

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()
test_true, test_pred, test_indices = [], [], []
with torch.inference_mode():
    for batch_images, batch_labels, batch_indices in loaders["test"]:
        logits = model(batch_images.to(device, non_blocking=True))
        test_true.extend(batch_labels.numpy())
        test_pred.extend(logits.argmax(1).cpu().numpy())
        test_indices.extend(batch_indices.numpy())
test_true, test_pred = np.asarray(test_true), np.asarray(test_pred)

print("TEST ACCURACY:", round(accuracy_score(test_true, test_pred), 4))
print("TEST MACRO F1:", round(f1_score(test_true, test_pred, average="macro"), 4))
report = pd.DataFrame(classification_report(
    test_true, test_pred, target_names=label_names, output_dict=True, zero_division=0
)).T
display(report.round(3))
report.to_csv(DRIVE_PROJECT_DIR / "resnet18_test_classification_report.csv")

results = metadata.iloc[test_indices].copy().reset_index(drop=True)
results["true_label"] = [label_names[i] for i in test_true]
results["predicted_label"] = [label_names[i] for i in test_pred]
results["correct"] = results["true_label"].eq(results["predicted_label"])
results.to_csv(DRIVE_PROJECT_DIR / "resnet18_test_predictions.csv", index=False)
display(results.groupby("collection_type")["correct"].agg(["count", "mean"]))

In [ ]:
fig, axis = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(test_true, test_pred), display_labels=label_names).plot(
    cmap="Blues", xticks_rotation=25, ax=axis, colorbar=False
)
axis.set_title("ResNet-18 Test Confusion Matrix")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "resnet18_test_confusion_matrix.png", dpi=160)
plt.show()
print("All model, history, report, prediction, and figure outputs were saved to Google Drive.")